# 랭체인(LangChain) Entity Extraction 예제 - 리뷰감정분석GPT 만들기(SentimentGPT)
## 작성자 : AISchool ( http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/ )
## 속성기반 감정분석 데이터 다운받기 : https://www.aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&aihubDataSe=data&dataSetSn=71603

# LangChain 라이브러리 설치

In [ ]:
!apt-get install -y jq libmagic-dev poppler-utils tesseract-ocr
!pip install -q -U langchain langchain-community langchain-openai langchain-chroma langchain-huggingface chromadb tiktoken pypdf unstructured sentence-transformers jq

# create_tagging_chain 살펴보기

## Reference : https://python.langchain.com/docs/use_cases/tagging

## OpenAI API Key 설정

In [ ]:
OPENAI_KEY = "Input Your Key"

In [ ]:
from langchain_openai import ChatOpenAI
import json

# 스키마 설정
schema = {
    "title": "TextTagging",
    "description": "Extract sentiment, aggressiveness, and language from the text.",
    "type": "object",
    "properties": {
        "sentiment": {  #감정 영역
            "type": "string",
            "description": "감정 상태 (예: 긍정, 부정, 중립)"
        },
        "aggressiveness": { # 공격성
            "type": "integer",
            "description": "공격성 점수 (1~10 사이의 점수)"
        },
        "language": { #사용된 언어
            "type": "string",
            "description": "주된 언어 (예: 한국어, 영어)"
        },
    },
    "required": ["sentiment", "aggressiveness", "language"]
}

#LLM 설정
llm = ChatOpenAI(
    temperature=0,
    model="gpt-4o-mini",
    api_key=OPENAI_KEY
)

# 체인 생성
chain = llm.with_structured_output(schema)

#입력 데이터 테스트
inp = "이 멍청아! 당장 내 돈 내놓으라고! 진짜 화가 머리 끝까지 나네."

# 실행 및 결과 출력
result = chain.invoke(inp)

print("--- [태깅 결과] ---")
print(json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
inp = "너를 만나게 되어 정말 기쁘다. 우리가 정말 좋은 친구가 될 거라고 생각해!"
chain.invoke(inp)

In [ ]:
inp = "너에게 정말 화가 났어! 너에게 마땅한 대가를 치르게 할 거야!"
chain.invoke(inp)

## enum값 지정하기

In [ ]:
schema = {
    "title": "ReviewSentimentAnalysis",
    "description": "Analyze the text for aggressiveness, language, and sentiment.",
    "type": "object",
    "properties": {
        "aggressiveness": { #공격성 판단 Entity
            "type": "integer",
            "enum": [1, 2, 3, 4, 5],  #값을 제한하여 설정
            "description": "describes how aggressive the statement is, the higher the number the more aggressive",
        },
        "language": { #언어 Entity
            "type": "string",
            "enum": ["spanish", "english", "french", "german", "italian", "korean"], #특정 언어를 제한하여 설정
        },
        "sentiment": { #감정 Entity
            "type": "string",
            "enum": ["positive", "negative"], #특정 감정을 제한하여 설정
        },
    },
    "required": ["language", "sentiment", "aggressiveness"],
}

In [ ]:
chain = llm.with_structured_output(schema)

In [ ]:
inp = "너를 만나서 정말 기쁘다! 우리가 정말 좋은 친구가 될 것 같아!"
chain.invoke(inp)

In [ ]:
inp = "너에게 정말 화가 나! 네가 받을 만한 것을 줄 거야!"
chain.invoke(inp)

In [ ]:
inp = "여기 날씨가 괜찮아, 외투만 입고 밖에 나갈 수 있어"
chain.invoke(inp)

# 리뷰감정분석GPT(SentimentGPT) 만들기

# 상품 리뷰 데이터 다운로드 & 업로드하기

In [ ]:
# /속성기반 감정분석 데이터/02.라벨링데이터/쇼핑몰/01. 패션/1-1. 여성의류/1-1.여성의류(1).json

## 데이터 셋 앞축 풀기

In [ ]:
!unzip -q -O cp949 -o "TL_쇼핑몰_01.패션_1-1.여성의류.zip.part0" -d ./shopping_data2

# 여성의류 리뷰 데이터 읽어오기

In [ ]:
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(
    file_path='./1-1.여성의류(1).json',
    jq_schema='.[]',
    text_content=False
)

docs = loader.load()
docs[0]

# 한글 인코딩 처리

In [ ]:
from langchain_core.documents import Document

refined_docs = []

for idx, doc in enumerate(docs):
    broken_korean = doc.page_content
    fixed_korean = broken_korean.encode('latin1').decode('unicode-escape')


    refined_doc = Document(
        page_content=fixed_korean,
        metadata=doc.metadata
    )
    refined_docs.append(refined_doc)


In [ ]:
len(refined_docs)

# 상품리뷰 데이터의 주요 Entity
*   **GeneralPolarity**	: 상품평 전체 감정 극성 (1: 긍정, -1: 부정)
*   **Aspects** : 상품평의 감정을 결정하는 단어와 분류카테고리(e.g. '착용감', '소재', '활용성'), 해당단어의 감정 극성(1: 긍정, -1: 부정), 감정을 결정하는 단어의 개수

In [ ]:
import json
json.loads(refined_docs[0].page_content)

In [ ]:
json.loads(refined_docs[99].page_content)

In [ ]:
# 전체 aspect 파악하기
aspect_sets = set()

for idx, doc in enumerate(refined_docs):
    data_json = json.loads(doc.page_content)
    aspects = data_json.get("Aspects")
    for aspect in aspects:
        aspect_sets.add(aspect["Aspect"])
print('aspect 개수 :', len(aspect_sets))
print(aspect_sets)

In [ ]:
schema = {
    "title": "SentimentAnalysis",
    "description": "Analyze the sentiment of a fashion product review.",
    "type": "object",
    "properties": {
        "sentiment word": {
            "type": "string",
            "description": "Please find the part of the word that expresses emotion in the whole sentence.",
        },
        "aspect": {
            "type": "string",
            "enum": [
                '가격', '기능', '길이', '두께', '디자인', '마감', '무게',
                '사이즈', '색상', '소재', '신축성', '제품구성', '착용감',
                '촉감', '품질', '핏', '활용성'
            ],
            "description": "The specific feature of the product discussed."
        },
        "sentiment": {
            "type": "string",
            "enum": ["positive", "negative", "neutral"],
            "description": "The emotional tone regarding the aspect."
        },
    },
    "required": ["aspect", "sentiment", "sentiment word"],
}

# 체인 생성 (create_tagging_chain 대체)
chain = llm.with_structured_output(schema)

## 10개 sampe에 대한 테스트

In [ ]:
chain.invoke(json.loads(refined_docs[0].page_content)['RawText'])

In [ ]:
json.loads(refined_docs[0].page_content)

In [ ]:
chain.invoke(json.loads(refined_docs[1].page_content)['RawText'])

In [ ]:
json.loads(refined_docs[1].page_content)

In [ ]:
chain.invoke(json.loads(refined_docs[2].page_content)['RawText'])

In [ ]:
json.loads(refined_docs[2].page_content)

In [ ]:
chain.invoke(json.loads(refined_docs[3].page_content)['RawText'])

In [ ]:
json.loads(refined_docs[3].page_content)

In [ ]:
chain.invoke(json.loads(refined_docs[4].page_content)['RawText'])

In [ ]:
json.loads(refined_docs[4].page_content)

In [ ]:
chain.invoke(json.loads(refined_docs[5].page_content)['RawText'])

In [ ]:
json.loads(refined_docs[5].page_content)

In [ ]:
chain.invoke(json.loads(refined_docs[6].page_content)['RawText'])

In [ ]:
json.loads(refined_docs[6].page_content)

In [ ]:
chain.invoke(json.loads(refined_docs[7].page_content)['RawText'])

In [ ]:
json.loads(refined_docs[7].page_content)

In [ ]:
chain.invoke(json.loads(refined_docs[8].page_content)['RawText'])

In [ ]:
json.loads(refined_docs[8].page_content)

In [ ]:
chain.invoke(json.loads(refined_docs[9].page_content)['RawText'])

In [ ]:
json.loads(refined_docs[9].page_content)